# BRIDGET: compas test



## Dataset Preprocessing


### Libraries, Retrieving data

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import yaml
import joblib

import pickle

import copy


import pandas as pd



from river import tree

from river import preprocessing
from river import ensemble, forest, compose


from sklearn.preprocessing import MinMaxScaler
from bridget_utils import *
from orchestrator import *
from master_config import *
from classes import BetaUser
from bridget_main import BRIDGET, HiC, MiC

"""
from baselines.compare_confidence import *
from baselines.differentiable_triage import *
from baselines.lce_surrogate import *
from baselines.mix_of_exps import *
from baselines.one_v_all import *
from baselines.selective_prediction import *
"""


26-Apr-02 23:37:16 fatf.utils.array.tools INFO     Using numpy's numpy.lib.recfunctions.structured_to_unstructured as fatf.utils.array.tools.structured_to_unstructured and fatf.utils.array.tools.structured_to_unstructured_row.


'\nfrom baselines.compare_confidence import *\nfrom baselines.differentiable_triage import *\nfrom baselines.lce_surrogate import *\nfrom baselines.mix_of_exps import *\nfrom baselines.one_v_all import *\nfrom baselines.selective_prediction import *\n'

In [3]:
set_all_seeds(42)

In [4]:
ds_name= 'compas'

In [5]:
data= pd.read_csv(fr".\datasets\{ds_name}.csv")
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14788 entries, 0 to 14787
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   sex              14788 non-null  object
 1   age              14788 non-null  int64 
 2   race             14788 non-null  object
 3   juv_fel_count    14788 non-null  int64 
 4   juv_misd_count   14788 non-null  int64 
 5   juv_other_count  14788 non-null  int64 
 6   priors_count     14788 non-null  int64 
 7   c_charge_degree  14788 non-null  object
 8   compas_score     14788 non-null  int64 
 9   did_recid        14788 non-null  int64 
dtypes: int64(7), object(3)
memory usage: 1.1+ MB


In [6]:
for c in data:
    print(data[c].value_counts().sum)

<bound method Series.sum of sex
Male      12093
Female     2695
Name: count, dtype: int64>
<bound method Series.sum of age
22    780
21    762
26    741
24    738
25    724
     ... 
96      2
78      1
83      1
80      1
79      1
Name: count, Length: 65, dtype: int64>
<bound method Series.sum of race
African-American    8004
Caucasian           4848
Hispanic            1083
Other                763
Asian                 57
Native American       33
Name: count, dtype: int64>
<bound method Series.sum of juv_fel_count
0     14176
1       390
2       135
3        46
4        19
5         9
8         4
10        4
6         3
20        1
13        1
Name: count, dtype: int64>
<bound method Series.sum of juv_misd_count
0     13854
1       643
2       174
3        57
4        23
8        11
6         9
5         8
12        4
7         3
13        2
Name: count, dtype: int64>
<bound method Series.sum of juv_other_count
0     13494
1       902
2       251
3        81
4        38
5         7

### Preprocessing Pipeline
1. Drop duplicates

2. Map 'sex', 'race', 'c_charge_degree'


In [7]:
data= clean_compas(data, 'compas_score', col_to_strip= 'c_charge_degree', drop_duplicates=True)

In [8]:
data.info()
data.head(n=5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5417 entries, 0 to 5416
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   sex              5417 non-null   object
 1   age              5417 non-null   int64 
 2   race             5417 non-null   object
 3   juv_fel_count    5417 non-null   int64 
 4   juv_misd_count   5417 non-null   int64 
 5   juv_other_count  5417 non-null   int64 
 6   priors_count     5417 non-null   int64 
 7   c_charge_degree  5417 non-null   object
 8   did_recid        5417 non-null   int64 
dtypes: int64(6), object(3)
memory usage: 381.0+ KB


,sex,age,race,juv_fel_count,juv_misd_count,juv_other_count,priors_count,c_charge_degree,did_recid
0,Female,19,African-American,0,0,0,0,F3,0
1,Female,19,African-American,0,0,0,0,M2,0
2,Female,19,African-American,0,0,0,0,M2,1
3,Female,19,African-American,0,0,0,1,F2,1
4,Female,19,African-American,0,0,0,1,F3,1


In [9]:
mapping= { 'sex': {'Female':0, 'Male':1},
           
            'race': {'Native American': 0,
            'Asian': 1,
            'Other': 2,
            'Hispanic': 3,
            'Caucasian': 4,
            'African-American': 5  },

            'c_charge_degree': {
                'X': 0, 'TCX': 0, 'NI0': 0, 'MO3': 0, 'CO3': 0, 

                'M2': 1,   
                'M1': 2,  
                'F5': 3, 'F6': 3,'F3': 3,                    
                                    
                'F7': 4, 'F2': 4, # apparently lvl 7 felonies can get u up to 15 years in prison                                                  
                'F1': 5
                }

}

data= apply_map(data, mapping)


### Splitting and Transforming data

1. Apply stratified sampling

2. Get pre-training/HiC/calibration/Mic data

3. Apply scaler

4. Get X, y

In [10]:
# Qui definiamo i vari split dei flussi 
set_all_seeds(42)
data = data.sample(frac=1, random_state=42).reset_index(drop=True) # shuffle iniziale

class_0 = data[data['did_recid'] == 0]
class_1= data[data['did_recid'] == 1]

#  split ufficiale

splits= {
    'calibration': (0.6, 0.8),
    'mic': (0.8, 1.0),
    'avv_train': (0.0, 0.07),
    'avv_test': (0.07, 0.1),
    'hic_train': (0.1, 0.5),
    'hic_test': (0.5, 0.6)
}

dfs= {}

for name, (start, end) in splits.items():
    dfs[name]= stratif(start, end, class_0, class_1)



In [11]:
for name, df in dfs.items():
    print(f"{name} length: {len(df)}")


calibration length: 1083
mic length: 1085
avv_train length: 378
avv_test length: 163
hic_train length: 2167
hic_test length: 541


In [12]:
target= 'did_recid'

categoricals= ['sex', 'race', 'c_charge_degree']
numericals= [c for c in data if c not in categoricals and c != target]


prepr_transf = (
    (compose.Select(*numericals, *categoricals) | preprocessing.MinMaxScaler())
)

In [13]:
## ora divisione in x e y

set_all_seeds(42)

# avviamento 
X_avv_train, y_avv_train = x_y_split(dfs['avv_train'], target)
X_avv_test, y_avv_test = x_y_split(dfs['avv_test'], target)


# hic
X_hic_train, y_hic_train = x_y_split(dfs['hic_train'], target)
X_hic_test, y_hic_test = x_y_split(dfs['hic_test'], target)

# validation
X_val, y_val = x_y_split(dfs['calibration'], target)

# mic
X_mic, y_mic = x_y_split(dfs['mic'], target)



## Calibration Phase: Experts and Incremental Model Selection

### Calibrating Incremental Model

The incremental model to be chosen for Bridget is trained on the X_avv, y_avv portion of the dataset,then evaluated on the X_avv_test and y_avv_test

The calibration phase starts by assessing the results of the learning for several configurations:

    - HoeffdingTreeClassifier

    - ExtremelyFastDecisionTreeClassifier

    - AdaBoostClassifier            (base= SGTClassifier)

    - AdwinBaggingClassifier        (base= SGTClassifier)

    - SRPClassifier                 (base= SGTClassifier)

    - AdaptiveRandomForestClassifier


The metrics observed are the Accuracy, the F1Score and the Counters for the classes

In [14]:
# since all River models work with dicts, lets first transform the dfs to dict
set_all_seeds(42)
X_avv_dict= X_avv_train.to_dict(orient='records')
X_avv_dict_test= X_avv_test.to_dict(orient='records')

df_batch_1 = (dfs['hic_train']).reset_index(drop=True)
mic_data= dfs['mic'].reset_index(drop=True)
df_avv= pd.concat([dfs['avv_train'], dfs['avv_test']]).reset_index(drop=True)

test_batch_1= X_hic_test.copy()
test_batch_1[target]= y_hic_test


# setting the init params required by HIC class
RULE = False
PAST = True
SKEPT = True
GROUP = True
EVA=    True
N_BINS = 10
N_VAR = 3
MAX = 5

rule_att = 'priors_count' #random rule
rule_value = -0.7

protected= ['race', 'age', 'sex']


In [15]:
# then the models are instantiated and trained by the HiC.train function
# the HiC object is initialized by passing a random user model, its not relevant since it won't interact with the IL anyways

set_all_seeds(42)

base = tree.HoeffdingTreeClassifier(grace_period=100)

htree= tree.HoeffdingAdaptiveTreeClassifier(grace_period= 100, seed= 42)
efdt= tree.ExtremelyFastDecisionTreeClassifier(grace_period=100)
ada= ensemble.AdaBoostClassifier(model= base, n_models= 15, seed= 42)  
adwin= ensemble.ADWINBaggingClassifier(model= base, n_models= 15, seed= 42)
srp= ensemble.SRPClassifier(model= base, n_models=15, seed= 42)
arf= forest.ARFClassifier(n_models= 15, grace_period= 100, max_features='sqrt', seed=42)

models= {
    'HoeffdingATC': htree, 
    "EFDT": efdt, 
    "AdaBoostCl":ada, 
    "ADWINBAGGING": adwin, 
    "SRPCL": srp, 
    "ARF":arf   
    }

prepr_dir= DATASETS[ds_name]['base_obj_paths']['preprocessors']
os.makedirs(prepr_dir, exist_ok=True)

model_dir= DATASETS[ds_name]['base_obj_paths']['incremental_learners']
os.makedirs(model_dir, exist_ok=True)

for model_name, model_obj in models.items():
    
    bridget_inst= HiC(RULE, PAST, SKEPT, GROUP, EVA, N_BINS, N_VAR, MAX, 
                rule_att, rule_value, 'placeholder', model_obj,
                start_performance= DATASETS[ds_name]['start_performance'],
                allocated_budget= DATASETS[ds_name]['allocated_budget'][0],
                skepticism_threshold= DATASETS[ds_name]['skepticism_threshold'],
                performance_delta= DATASETS[ds_name]['performance_delta'],
                dataset_name= ds_name,
                user_name= 'placeholder',
                batch1=df_batch_1, batch3=mic_data, batch1_test=test_batch_1, 
                target=target, 
                user_model='placeholder', 
                protected=protected, cats=categoricals, num=numericals,
                preprocessor=prepr_transf,
                training_iter= 0 
                )
    
    bridget_inst.train(X_avv_dict, y_avv_train, X_avv_dict_test, y_avv_test)

    base_preprocessor = bridget_inst.preprocessor
    base_model = bridget_inst.hic_model
    

    prepr_filename = f"{model_name}_preprocessor.pkl"
    model_filename = f"{model_name}_model.pkl"

    joblib.dump(base_preprocessor, os.path.join(prepr_dir, prepr_filename))
    joblib.dump(base_model, os.path.join(model_dir, model_filename))


trained_arf= arf


Accuracy: 65.03%
F1: 42.42%
Distribution of predictions: Counter({0: 131, 1: 32})
HoeffdingAdaptiveTreeClassifier trained
Accuracy: 58.90%
F1: 0.00%
Distribution of predictions: Counter({0: 163})
ExtremelyFastDecisionTreeClassifier trained
Accuracy: 66.87%
F1: 46.00%
Distribution of predictions: Counter({0: 130, 1: 33})
AdaBoostClassifier(HoeffdingTreeClassifier) trained
Accuracy: 66.26%
F1: 43.30%
Distribution of predictions: Counter({0: 133, 1: 30})
ADWINBaggingClassifier(HoeffdingTreeClassifier) trained
Accuracy: 63.80%
F1: 39.18%
Distribution of predictions: Counter({0: 133, 1: 30})
SRPClassifier(HoeffdingTreeClassifier) trained
Accuracy: 64.42%
F1: 40.82%
Distribution of predictions: Counter({0: 132, 1: 31})
ARFClassifier trained


### Calibrating Experts


In [16]:
with open(fr".\experts_{ds_name}.yaml", "r") as f:
    config= yaml.safe_load(f)


params_dict= config['experts']['groups']['w_dict']


In [17]:
set_all_seeds(42)
X_exp= X_hic_train.to_dict(orient='records')

X_exp_scaled= []

for x in X_exp:
    X_exp_scaled.append(prepr_transf.transform_one(x))

X_exp_final = pd.DataFrame(X_exp_scaled)

In [18]:
set_all_seeds(42)
experts_obj= {}

expert_names = ['accurate_trusting', 'accurate_not_trusting', 
                'inaccurate_trusting', 'inaccurate_not_trusting']

for name in expert_names:
    expert_type= config['experts']['groups'][name]

    experts_obj[name]= BetaUser(
        belief_level= expert_type['belief_value'],
        rethink_level= 0.8, # as suggested by the og FRANK implementation
        fairness= True,
        fpr= expert_type['target_FPR'],
        fnr= expert_type['target_FNR'],
        alpha= 0.9,
        features_dict= params_dict,
        seed= expert_type['group_seed']
        )
    res = experts_obj[name].fit(X_exp_final, y_hic_train, tol= 0.001)

    save_dir= fr".\trained_experts\{ds_name}"
    os.makedirs(save_dir, exist_ok= True)
    file_path = os.path.join(save_dir, f"{name}.pkl")

    with open(file_path, 'wb') as f:
        pickle.dump(experts_obj[name], f)
    print(f"Expert saved in: {file_path}")

    
    print(f"{'='*30}")
    print(f" EXPERT CALIBRATION REPORT ")
    print(f"{'='*30}")

    print(f"\n[EXPERT: {name}]")
    print(f"\n[FALSE POSITIVE RATE]")
    print(f"  - Iters:      {res['fpr iters number']}")
    print(f"  - Beta:       {res['calibrated_fpr_beta']:.4f}")
    print(f"  - Target:     {res['target_fpr']}")
    print(f"  - Achieved:   {res['achieved_fpr']:.4f}")

    print(f"\n[FALSE NEGATIVE RATE]")
    print(f"  - Iters:      {res['fnr iters number']}")
    print(f"  - Beta:       {res['calibrated_fnr_beta']:.4f}")
    print(f"  - Target:     {res['target_fnr']}")
    print(f"  - Achieved:   {res['achieved_fnr']:.4f}")
    


Expert saved in: .\trained_experts\compas\accurate_trusting.pkl
 EXPERT CALIBRATION REPORT 

[EXPERT: accurate_trusting]

[FALSE POSITIVE RATE]
  - Iters:      11
  - Beta:       -3.2227
  - Target:     0.05
  - Achieved:   0.0498

[FALSE NEGATIVE RATE]
  - Iters:      13
  - Beta:       -2.6123
  - Target:     0.05
  - Achieved:   0.0501
Expert saved in: .\trained_experts\compas\accurate_not_trusting.pkl
 EXPERT CALIBRATION REPORT 

[EXPERT: accurate_not_trusting]

[FALSE POSITIVE RATE]
  - Iters:      12
  - Beta:       -3.0762
  - Target:     0.05
  - Achieved:   0.0491

[FALSE NEGATIVE RATE]
  - Iters:      12
  - Beta:       -2.7832
  - Target:     0.05
  - Achieved:   0.0506
Expert saved in: .\trained_experts\compas\inaccurate_trusting.pkl
 EXPERT CALIBRATION REPORT 

[EXPERT: inaccurate_trusting]

[FALSE POSITIVE RATE]
  - Iters:      14
  - Beta:       -1.2329
  - Target:     0.3
  - Achieved:   0.3008

[FALSE NEGATIVE RATE]
  - Iters:      14
  - Beta:       -0.3784
  - Target

## BRIDGET decision making



In [19]:
# base params for all users
initial_ordering= [c for c in data if c != target]

# retrieve preprocessor and incremental learner
rules= FRANK_RULES


base_params= {
    "cats": categoricals, #lst
    "num": numericals, #lss
    "warm_up_set": df_avv.copy(),
    "batch1": df_batch_1.copy(),
    "batch1_test": test_batch_1.copy(),
    "batch3":dfs['mic'].copy(),
    "validation_set": dfs['calibration'].copy(), #obtained by hic
    "feature_order": initial_ordering.copy(), #without the label ?
    "incremental_learner_name":"ARF",
    "n_cols": len(initial_ordering), #obtained by hic
    "mic_model_name": "Def_Net",
    "strat_1_name": "Confidence",
    "strat_2_name": "Mao",
    
}


learner_path= DATASETS[ds_name]['base_obj_paths']['incremental_learners']
path= os.path.join(learner_path, f"{base_params["incremental_learner_name"]}_model.pkl")
trained_learner= joblib.load(path)

prepr_path= DATASETS[ds_name]['base_obj_paths']['preprocessors']
path= os.path.join(prepr_path, f"{base_params["incremental_learner_name"]}_preprocessor.pkl")
base_prepr= joblib.load(path)

base_objects = {
      "preprocessor": copy.deepcopy(base_prepr), #or take from path
      "incremental_learner":copy.deepcopy(trained_learner), # or take from path
      "scaler": None
} 

#device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device= torch.device("cpu")
#add these once you get to mic phase: "run_confidence": True, "run_mao": True

### Expert: Accurate, Trusting 

#### HIC

In [20]:
#setting up fixed params

current_user_name= "accurate_trusting"
user_suffix= 'acc_t'

exp_path= fr".\trained_experts\{ds_name}\{name}.pkl"
current_expert= joblib.load(exp_path)


params = base_params.copy()  # Start with the base
params.update({
    "user_name": current_user_name,
    "user_suffix": user_suffix
})

objects = base_objects.copy() # Start with the base clones
objects.update({
    "user_model": current_expert
})

In [21]:
run_hic(ds_name, params, objects)

100%|██████████| 722/722 [01:26<00:00,  8.35it/s]


0.93359375 1.0765765765765767


2026-04-02 23:38:56,326 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training


Training Results - Epoch[1] Avg accuracy: 0.46 Avg loss: 0.71
Validation Results - Epoch[1] Avg accuracy: 0.41 Avg loss: 0.72
End of Epoch 1: Learning Rate 0.001
Training Results - Epoch[2] Avg accuracy: 0.46 Avg loss: 0.70
Validation Results - Epoch[2] Avg accuracy: 0.41 Avg loss: 0.72
End of Epoch 2: Learning Rate 0.001
Training Results - Epoch[3] Avg accuracy: 0.46 Avg loss: 0.70
Validation Results - Epoch[3] Avg accuracy: 0.41 Avg loss: 0.71
End of Epoch 3: Learning Rate 0.001
Training Results - Epoch[4] Avg accuracy: 0.46 Avg loss: 0.70
Validation Results - Epoch[4] Avg accuracy: 0.41 Avg loss: 0.71
End of Epoch 4: Learning Rate 0.001


c:\Users\virgm\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\virgm\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\virgm\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


...

1.1672131147540983 0.8746928746928747
Training Results - Epoch[1] Avg accuracy: 0.57 Avg loss: 0.71
Validation Results - Epoch[1] Avg accuracy: 0.41 Avg loss: 0.75
End of Epoch 1: Learning Rate 0.001
Training Results - Epoch[2] Avg accuracy: 0.57 Avg loss: 0.70
Validation Results - Epoch[2] Avg accuracy: 0.41 Avg loss: 0.74
End of Epoch 2: Learning Rate 0.001
Training Results - Epoch[3] Avg accuracy: 0.57 Avg loss: 0.70
Validation Results - Epoch[3] Avg accuracy: 0.41 Avg loss: 0.74
End of Epoch 3: Learning Rate 0.001
Training Results - Epoch[4] Avg accuracy: 0.57 Avg loss: 0.70
Validation Results - Epoch[4] Avg accuracy: 0.41 Avg loss: 0.73
End of Epoch 4: Learning Rate 0.001


c:\Users\virgm\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\virgm\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\virgm\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


1.1672131147540983 0.8746928746928747
Training Results - Epoch[1] Avg accuracy: 0.57 Avg loss: 0.69
Validation Results - Epoch[1] Avg accuracy: 0.41 Avg loss: 0.71
End of Epoch 1: Learning Rate 0.001
Training Results - Epoch[2] Avg accuracy: 0.58 Avg loss: 0.69
Validation Results - Epoch[2] Avg accuracy: 0.41 Avg loss: 0.71
End of Epoch 2: Learning Rate 0.001
Training Results - Epoch[3] Avg accuracy: 0.62 Avg loss: 0.69
Validation Results - Epoch[3] Avg accuracy: 0.47 Avg loss: 0.71
End of Epoch 3: Learning Rate 0.001
Training Results - Epoch[4] Avg accuracy: 0.65 Avg loss: 0.68
Validation Results - Epoch[4] Avg accuracy: 0.49 Avg loss: 0.70
End of Epoch 4: Learning Rate 0.001
Training Results - Epoch[5] Avg accuracy: 0.66 Avg loss: 0.68
Validation Results - Epoch[5] Avg accuracy: 0.49 Avg loss: 0.69
End of Epoch 5: Learning Rate 0.001
Training Results - Epoch[6] Avg accuracy: 0.66 Avg loss: 0.68
Validation Results - Epoch[6] Avg accuracy: 0.51 Avg loss: 0.69
End of Epoch 6: Learning R

2026-04-02 23:42:24,238 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training


Validation Results - Epoch[8] Avg accuracy: 0.49 Avg loss: 0.70
End of Epoch 8: Learning Rate 0.0008
Training Results - Epoch[9] Avg accuracy: 0.66 Avg loss: 0.66
Validation Results - Epoch[9] Avg accuracy: 0.49 Avg loss: 0.70
End of Epoch 9: Learning Rate 0.0008
Training Results - Epoch[10] Avg accuracy: 0.66 Avg loss: 0.65
Validation Results - Epoch[10] Avg accuracy: 0.49 Avg loss: 0.69
End of Epoch 10: Learning Rate 0.0008
1.1672131147540983 0.8746928746928747
Training Results - Epoch[1] Avg accuracy: 0.63 Avg loss: 0.68
Validation Results - Epoch[1] Avg accuracy: 0.43 Avg loss: 0.70
End of Epoch 1: Learning Rate 0.001
Training Results - Epoch[2] Avg accuracy: 0.66 Avg loss: 0.67
Validation Results - Epoch[2] Avg accuracy: 0.49 Avg loss: 0.70
End of Epoch 2: Learning Rate 0.001
Training Results - Epoch[3] Avg accuracy: 0.66 Avg loss: 0.66
Validation Results - Epoch[3] Avg accuracy: 0.49 Avg loss: 0.70
End of Epoch 3: Learning Rate 0.001
Training Results - Epoch[4] Avg accuracy: 0.66

#### CALIB

In [22]:
#chosen nets: "large", 32/16

net_layers= [32,16]
hic_df_general_path= DATASETS[ds_name]['paths']['hic_df_save_path']
net_general_path= DATASETS[ds_name]['paths']['def_net_save_path']

# retrieving df and best net path
for i in range (1, 4):

    df_switch_path= os.path.join(hic_df_general_path,
                            f"iter_{i}",
                            f"results_{params['user_name']}",
                            f"hic_{params['user_name']}.parquet"
                            )
    
    df_switch= pd.read_parquet(df_switch_path)


    def_net_path= os.path.join(net_general_path,
                           f"iter_{i}",
                           f"{user_suffix}_models",
                           f"{net_layers[0]}_{net_layers[1]}"
                          )

    pattern = f"{net_layers[0]}_{net_layers[1]}_model_*.pt"
    search_query = os.path.join(def_net_path, pattern)
    matches = glob.glob(search_query)

    if matches:
        def_net_path = max(matches, key=os.path.getctime)
        run_calibration(def_net_path= def_net_path, 
                        df_switch=df_switch, 
                        ds_name=ds_name, 
                        params=params,
                        device=device, 
                        iteration=i,
                        def_net_layers=net_layers
                        ) #default had batch seize 128, epochs = 20 

#### MIC

In [23]:
current_user_name= "accurate_trusting"
user_suffix= 'acc_t'

exp_path= fr".\trained_experts\{ds_name}\{name}.pkl"
current_expert= joblib.load(exp_path)


params = base_params.copy()  # Start with the base
params.update({
    "user_name": current_user_name,
    "user_suffix": user_suffix,
    "run_confidence": True,
    "run_mao": True
})

objects = base_objects.copy() # Start with the base clones
objects.update({
    "user_model": current_expert
})


def_net_layers= [32, 16]
strat_1=None
strat_2=None

for i in range(1,4):

    strat_1, strat_2= run_mic( 
                            ds_name= ds_name,
                            device= device,
                            layers= def_net_layers,
                            params=params,
                            objects=objects,
                            iteration=i,
                            strat_1_res=strat_1,
                            strat_2_res=strat_2
                            )
    
  




 25%|██▍       | 270/1085 [00:26<01:39,  8.18it/s]

2026-04-02 23:43:00,465 | WARNING | Drift here! Record index 270
2026-04-02 23:43:00,480 | WARNING | DRIFT DETECTED | Phase: MiC | Strat: Confidence


 25%|██▍       | 270/1085 [00:26<01:21, 10.05it/s]
c:\Bridget\orchestrator.py:953: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  anqi_mao_thresh= anqi_mao_thresh_row[0]
 25%|██▍       | 270/1085 [00:25<01:34,  8.65it/s]

2026-04-02 23:43:26,105 | WARNING | Drift here! Record index 270
2026-04-02 23:43:26,121 | WARNING | DRIFT DETECTED | Phase: MiC | Strat: Mao


 25%|██▍       | 270/1085 [00:25<01:17, 10.54it/s]
c:\Bridget\orchestrator.py:953: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  anqi_mao_thresh= anqi_mao_thresh_row[0]
 25%|██▍       | 270/1085 [00:24<01:28,  9.18it/s]

...

 25%|██▍       | 270/1085 [00:30<01:32,  8.85it/s]
c:\Bridget\orchestrator.py:953: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  anqi_mao_thresh= anqi_mao_thresh_row[0]
 25%|██▍       | 270/1085 [00:29<02:05,  6.47it/s]

2026-04-02 23:58:38,221 | WARNING | Drift here! Record index 270
2026-04-02 23:58:38,235 | WARNING | DRIFT DETECTED | Phase: MiC | Strat: Mao


 25%|██▍       | 270/1085 [00:29<01:28,  9.19it/s]
c:\Bridget\orchestrator.py:953: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  anqi_mao_thresh= anqi_mao_thresh_row[0]
 25%|██▍       | 270/1085 [00:30<01:49,  7.45it/s]

2026-04-02 23:59:08,737 | WARNING | Drift here! Record index 270
2026-04-02 23:59:08,752 | WARNING | DRIFT DETECTED | Phase: MiC | Strat: Mao


 25%|██▍       | 270/1085 [00:30<01:31,  8.86it/s]


### Expert: Inaccurate, Trusting 

#### HIC

In [24]:
#setting up fixed params

current_user_name= "inaccurate_trusting"
user_suffix= 'inacc_t'

exp_path= fr".\trained_experts\{ds_name}\{name}.pkl"
current_expert= joblib.load(exp_path)


params = base_params.copy()  # Start with the base
params.update({
    "user_name": current_user_name,
    "user_suffix": user_suffix
})

objects = base_objects.copy() # Start with the base clones
objects.update({
    "user_model": current_expert
})

In [25]:
run_hic(ds_name, params, objects)
       

100%|██████████| 722/722 [01:55<00:00,  6.25it/s]
2026-03-27 16:50:26,303 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training


1.126602564102564 0.8989769820971867
Training Results - Epoch[1] Avg accuracy: 0.56 Avg loss: 0.71
Validation Results - Epoch[1] Avg accuracy: 0.41 Avg loss: 0.75
End of Epoch 1: Learning Rate 0.001
Training Results - Epoch[2] Avg accuracy: 0.56 Avg loss: 0.70
Validation Results - Epoch[2] Avg accuracy: 0.41 Avg loss: 0.74
End of Epoch 2: Learning Rate 0.001
Training Results - Epoch[3] Avg accuracy: 0.56 Avg loss: 0.70
Validation Results - Epoch[3] Avg accuracy: 0.41 Avg loss: 0.73
End of Epoch 3: Learning Rate 0.001
Training Results - Epoch[4] Avg accuracy: 0.56 Avg loss: 0.70
Validation Results - Epoch[4] Avg accuracy: 0.41 Avg loss: 0.73
End of Epoch 4: Learning Rate 0.001


c:\Users\virgm\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\virgm\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\virgm\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


1.126602564102564 0.8989769820971867
Training Results - Epoch[1] Avg accuracy: 0.56 Avg loss: 0.69
Validation Results - Epoch[1] Avg accuracy: 0.41 Avg loss: 0.72
End of Epoch 1: Learning Rate 0.001
Training Results - Epoch[2] Avg accuracy: 0.56 Avg loss: 0.69
Validation Results - Epoch[2] Avg accuracy: 0.41 Avg loss: 0.71
End of Epoch 2: Learning Rate 0.001
Training Results - Epoch[3] Avg accuracy: 0.56 Avg loss: 0.69
Validation Results - Epoch[3] Avg accuracy: 0.41 Avg loss: 0.71
End of Epoch 3: Learning Rate 0.001
Training Results - Epoch[4] Avg accuracy: 0.58 Avg loss: 0.69
Validation Results - Epoch[4] Avg accuracy: 0.42 Avg loss: 0.70
End of Epoch 4: Learning Rate 0.001
Training Results - Epoch[5] Avg accuracy: 0.58 Avg loss: 0.69
Validation Results - Epoch[5] Avg accuracy: 0.49 Avg loss: 0.70
End of Epoch 5: Learning Rate 0.001
Training Results - Epoch[6] Avg accuracy: 0.59 Avg loss: 0.69
Validation Results - Epoch[6] Avg accuracy: 0.51 Avg loss: 0.69
End of Epoch 6: Learning Ra

2026-03-27 16:50:27,925 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training


...

1.771144278606965 0.6966731898238747
Training Results - Epoch[1] Avg accuracy: 0.72 Avg loss: 0.71
Validation Results - Epoch[1] Avg accuracy: 0.41 Avg loss: 0.79
End of Epoch 1: Learning Rate 0.001
Training Results - Epoch[2] Avg accuracy: 0.72 Avg loss: 0.70
Validation Results - Epoch[2] Avg accuracy: 0.41 Avg loss: 0.78
End of Epoch 2: Learning Rate 0.001
Training Results - Epoch[3] Avg accuracy: 0.72 Avg loss: 0.70
Validation Results - Epoch[3] Avg accuracy: 0.41 Avg loss: 0.77
End of Epoch 3: Learning Rate 0.001
Training Results - Epoch[4] Avg accuracy: 0.72 Avg loss: 0.70
Validation Results - Epoch[4] Avg accuracy: 0.41 Avg loss: 0.77
End of Epoch 4: Learning Rate 0.001


c:\Users\virgm\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\virgm\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\virgm\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


1.771144278606965 0.6966731898238747
Training Results - Epoch[1] Avg accuracy: 0.72 Avg loss: 0.69
Validation Results - Epoch[1] Avg accuracy: 0.41 Avg loss: 0.74
End of Epoch 1: Learning Rate 0.001
Training Results - Epoch[2] Avg accuracy: 0.72 Avg loss: 0.69
Validation Results - Epoch[2] Avg accuracy: 0.41 Avg loss: 0.73
End of Epoch 2: Learning Rate 0.001
Training Results - Epoch[3] Avg accuracy: 0.74 Avg loss: 0.68
Validation Results - Epoch[3] Avg accuracy: 0.45 Avg loss: 0.73
End of Epoch 3: Learning Rate 0.001
Training Results - Epoch[4] Avg accuracy: 0.75 Avg loss: 0.68
Validation Results - Epoch[4] Avg accuracy: 0.49 Avg loss: 0.72
End of Epoch 4: Learning Rate 0.001
Training Results - Epoch[5] Avg accuracy: 0.75 Avg loss: 0.67
Validation Results - Epoch[5] Avg accuracy: 0.49 Avg loss: 0.71
End of Epoch 5: Learning Rate 0.001
Training Results - Epoch[6] Avg accuracy: 0.76 Avg loss: 0.67
Validation Results - Epoch[6] Avg accuracy: 0.49 Avg loss: 0.71
End of Epoch 6: Learning Ra

2026-03-27 16:54:07,459 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training


1.771144278606965 0.6966731898238747
Training Results - Epoch[1] Avg accuracy: 0.72 Avg loss: 0.68
Validation Results - Epoch[1] Avg accuracy: 0.41 Avg loss: 0.73
End of Epoch 1: Learning Rate 0.001
Training Results - Epoch[2] Avg accuracy: 0.76 Avg loss: 0.66
Validation Results - Epoch[2] Avg accuracy: 0.49 Avg loss: 0.73
End of Epoch 2: Learning Rate 0.001
Training Results - Epoch[3] Avg accuracy: 0.76 Avg loss: 0.65
Validation Results - Epoch[3] Avg accuracy: 0.49 Avg loss: 0.73
End of Epoch 3: Learning Rate 0.001
Training Results - Epoch[4] Avg accuracy: 0.76 Avg loss: 0.63
Validation Results - Epoch[4] Avg accuracy: 0.49 Avg loss: 0.73
End of Epoch 4: Learning Rate 0.001
Training Results - Epoch[5] Avg accuracy: 0.76 Avg loss: 0.61
Validation Results - Epoch[5] Avg accuracy: 0.49 Avg loss: 0.74
End of Epoch 5: Learning Rate 0.001


#### Calib

In [26]:
#chosen nets: "large", 32/16

layers= [32,16]
hic_df_general_path= DATASETS[ds_name]['paths']['hic_df_save_path']
net_general_path= DATASETS[ds_name]['paths']['def_net_save_path']
# retrieving df and best net path

for i in range (1, 4):

    df_switch_path= os.path.join(hic_df_general_path,
                            f"iter_{i}",
                            f"results_{params['user_name']}",
                            f"hic_{params['user_name']}.parquet"
                            )
    
    df_switch= pd.read_parquet(df_switch_path)


    def_net_path= os.path.join(net_general_path,
                           f"iter_{i}",
                           f"{user_suffix}_models",
                           f"{layers[0]}_{layers[1]}"
                          )

    pattern = f"{layers[0]}_{layers[1]}_model_*.pt"
    search_query = os.path.join(def_net_path, pattern)
    matches = glob.glob(search_query)

    if matches:
        def_net_path = max(matches, key=os.path.getctime)
        run_calibration(def_net_path= def_net_path, 
                        df_switch=df_switch, 
                        ds_name=ds_name, 
                        params=params,
                        device=device, 
                        iteration=i,
                        def_net_layers=net_layers
                        ) #default had batch seize 128, epochs = 20 

#### MIC

In [27]:
current_user_name= "inaccurate_trusting"
user_suffix= 'inacc_t'

exp_path= fr".\trained_experts\{ds_name}\{name}.pkl"
current_expert= joblib.load(exp_path)


params = base_params.copy()  # Start with the base
params.update({
    "user_name": current_user_name,
    "user_suffix": user_suffix,
    "run_confidence": True,
    "run_mao": True
})

objects = base_objects.copy() # Start with the base clones
objects.update({
    "user_model": current_expert
})


def_net_layers= [32, 16]
strat_1=None
strat_2=None

for i in range(1,4):

    strat_1, strat_2= run_mic( 
                            ds_name= ds_name,
                            device= device,
                            layers= def_net_layers,
                            params=params,
                            objects=objects,
                            iteration=i,
                            strat_1_res=strat_1,
                            strat_2_res=strat_2)
    
  

100%|██████████| 1085/1085 [01:25<00:00, 12.66it/s]
c:\Bridget\orchestrator.py:953: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  anqi_mao_thresh= anqi_mao_thresh_row[0]
 25%|██▍       | 270/1085 [00:21<01:04, 12.72it/s]

2026-03-27 16:56:00,870 | WARNING | Drift here! Record index 270
2026-03-27 16:56:00,885 | WARNING | DRIFT DETECTED | Phase: MiC | Strat: Mao


 25%|██▍       | 270/1085 [00:21<01:05, 12.48it/s]
c:\Bridget\orchestrator.py:953: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  anqi_mao_thresh= anqi_mao_thresh_row[0]
 25%|██▍       | 270/1085 [00:21<01:01, 13.27it/s]

2026-03-27 16:56:22,455 | WARNING | Drift here! Record index 270
2026-03-27 16:56:22,472 | WARNING | DRIFT DETECTED | Phase: MiC | Strat: Mao


 25%|██▍       | 270/1085 [00:21<01:05, 12.53it/s]
c:\Bridget\orchestrator.py:953: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  anqi_mao_thresh= anqi_mao_thresh_row[0]
 25%|██▍       | 270/1085 [00:21<01:01, 13.30it/s]

...

 25%|██▍       | 270/1085 [00:23<01:12, 11.25it/s]
c:\Bridget\orchestrator.py:953: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  anqi_mao_thresh= anqi_mao_thresh_row[0]
 25%|██▍       | 269/1085 [00:23<01:11, 11.36it/s]

2026-03-27 17:03:02,807 | WARNING | Drift here! Record index 270
2026-03-27 17:03:02,825 | WARNING | DRIFT DETECTED | Phase: MiC | Strat: Mao


 25%|██▍       | 270/1085 [00:23<01:11, 11.33it/s]
c:\Bridget\orchestrator.py:953: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  anqi_mao_thresh= anqi_mao_thresh_row[0]
 25%|██▍       | 269/1085 [00:23<01:16, 10.73it/s]

2026-03-27 17:03:27,143 | WARNING | Drift here! Record index 270
2026-03-27 17:03:27,159 | WARNING | DRIFT DETECTED | Phase: MiC | Strat: Mao


 25%|██▍       | 270/1085 [00:24<01:13, 11.11it/s]


### Expert: Accurate, Not Trusting 

#### HIC

In [28]:
#setting up fixed params

current_user_name= "accurate_not_trusting"
user_suffix= 'acc_nt'

exp_path= fr".\trained_experts\{ds_name}\{name}.pkl"
current_expert= joblib.load(exp_path)


params = base_params.copy()  # Start with the base
params.update({
    "user_name": current_user_name,
    "user_suffix": user_suffix
})

objects = base_objects.copy() # Start with the base clones
objects.update({
    "user_model": current_expert
})

In [29]:
run_hic(ds_name, params, objects)

100%|██████████| 722/722 [01:25<00:00,  8.44it/s]
2026-03-27 17:04:53,168 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training


1.1716666666666666 0.8722084367245657
Training Results - Epoch[1] Avg accuracy: 0.57 Avg loss: 0.71
Validation Results - Epoch[1] Avg accuracy: 0.41 Avg loss: 0.75
End of Epoch 1: Learning Rate 0.001
Training Results - Epoch[2] Avg accuracy: 0.57 Avg loss: 0.70
Validation Results - Epoch[2] Avg accuracy: 0.41 Avg loss: 0.74
End of Epoch 2: Learning Rate 0.001
Training Results - Epoch[3] Avg accuracy: 0.57 Avg loss: 0.70
Validation Results - Epoch[3] Avg accuracy: 0.41 Avg loss: 0.74
End of Epoch 3: Learning Rate 0.001
Training Results - Epoch[4] Avg accuracy: 0.57 Avg loss: 0.70
Validation Results - Epoch[4] Avg accuracy: 0.41 Avg loss: 0.73
End of Epoch 4: Learning Rate 0.001


c:\Users\virgm\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\virgm\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\virgm\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


1.1716666666666666 0.8722084367245657
Training Results - Epoch[1] Avg accuracy: 0.57 Avg loss: 0.69
Validation Results - Epoch[1] Avg accuracy: 0.41 Avg loss: 0.71
End of Epoch 1: Learning Rate 0.001
Training Results - Epoch[2] Avg accuracy: 0.58 Avg loss: 0.69
Validation Results - Epoch[2] Avg accuracy: 0.41 Avg loss: 0.71
End of Epoch 2: Learning Rate 0.001
Training Results - Epoch[3] Avg accuracy: 0.60 Avg loss: 0.69
Validation Results - Epoch[3] Avg accuracy: 0.45 Avg loss: 0.71
End of Epoch 3: Learning Rate 0.001
Training Results - Epoch[4] Avg accuracy: 0.62 Avg loss: 0.69
Validation Results - Epoch[4] Avg accuracy: 0.49 Avg loss: 0.70
End of Epoch 4: Learning Rate 0.001
Training Results - Epoch[5] Avg accuracy: 0.62 Avg loss: 0.68
Validation Results - Epoch[5] Avg accuracy: 0.49 Avg loss: 0.70
End of Epoch 5: Learning Rate 0.001
Training Results - Epoch[6] Avg accuracy: 0.63 Avg loss: 0.68
Validation Results - Epoch[6] Avg accuracy: 0.51 Avg loss: 0.69
End of Epoch 6: Learning R

2026-03-27 17:04:55,399 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training


...

1.934782608695652 0.6742424242424242
Training Results - Epoch[1] Avg accuracy: 0.74 Avg loss: 0.69
Validation Results - Epoch[1] Avg accuracy: 0.41 Avg loss: 0.74
End of Epoch 1: Learning Rate 0.001
Training Results - Epoch[2] Avg accuracy: 0.74 Avg loss: 0.68
Validation Results - Epoch[2] Avg accuracy: 0.41 Avg loss: 0.74
End of Epoch 2: Learning Rate 0.001
Training Results - Epoch[3] Avg accuracy: 0.77 Avg loss: 0.68
Validation Results - Epoch[3] Avg accuracy: 0.46 Avg loss: 0.73
End of Epoch 3: Learning Rate 0.001
Training Results - Epoch[4] Avg accuracy: 0.79 Avg loss: 0.67
Validation Results - Epoch[4] Avg accuracy: 0.49 Avg loss: 0.72
End of Epoch 4: Learning Rate 0.001
Training Results - Epoch[5] Avg accuracy: 0.79 Avg loss: 0.67
Validation Results - Epoch[5] Avg accuracy: 0.49 Avg loss: 0.72
End of Epoch 5: Learning Rate 0.001
Training Results - Epoch[6] Avg accuracy: 0.80 Avg loss: 0.66
Validation Results - Epoch[6] Avg accuracy: 0.49 Avg loss: 0.71
End of Epoch 6: Learning Ra

2026-03-27 17:07:53,645 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training


Training Results - Epoch[9] Avg accuracy: 0.80 Avg loss: 0.64
Validation Results - Epoch[9] Avg accuracy: 0.49 Avg loss: 0.71
End of Epoch 9: Learning Rate 0.001
1.934782608695652 0.6742424242424242
Training Results - Epoch[1] Avg accuracy: 0.74 Avg loss: 0.69
Validation Results - Epoch[1] Avg accuracy: 0.41 Avg loss: 0.77
End of Epoch 1: Learning Rate 0.001
Training Results - Epoch[2] Avg accuracy: 0.74 Avg loss: 0.69
Validation Results - Epoch[2] Avg accuracy: 0.41 Avg loss: 0.76
End of Epoch 2: Learning Rate 0.001
Training Results - Epoch[3] Avg accuracy: 0.74 Avg loss: 0.68
Validation Results - Epoch[3] Avg accuracy: 0.41 Avg loss: 0.75
End of Epoch 3: Learning Rate 0.001
Training Results - Epoch[4] Avg accuracy: 0.78 Avg loss: 0.67
Validation Results - Epoch[4] Avg accuracy: 0.48 Avg loss: 0.75
End of Epoch 4: Learning Rate 0.001
Training Results - Epoch[5] Avg accuracy: 0.79 Avg loss: 0.66
Validation Results - Epoch[5] Avg accuracy: 0.48 Avg loss: 0.75
End of Epoch 5: Learning Ra

2026-03-27 17:07:54,211 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training


Validation Results - Epoch[9] Avg accuracy: 0.49 Avg loss: 0.76
End of Epoch 9: Learning Rate 0.0008
Training Results - Epoch[10] Avg accuracy: 0.80 Avg loss: 0.61
Validation Results - Epoch[10] Avg accuracy: 0.49 Avg loss: 0.76
End of Epoch 10: Learning Rate 0.0008
Training Results - Epoch[11] Avg accuracy: 0.80 Avg loss: 0.60
Validation Results - Epoch[11] Avg accuracy: 0.49 Avg loss: 0.76
End of Epoch 11: Learning Rate 0.0008
Training Results - Epoch[12] Avg accuracy: 0.80 Avg loss: 0.59
Validation Results - Epoch[12] Avg accuracy: 0.49 Avg loss: 0.77
End of Epoch 12: Learning Rate 0.0008
Training Results - Epoch[13] Avg accuracy: 0.80 Avg loss: 0.58
Validation Results - Epoch[13] Avg accuracy: 0.49 Avg loss: 0.77
End of Epoch 13: Learning Rate 0.0008
Training Results - Epoch[14] Avg accuracy: 0.80 Avg loss: 0.57
Validation Results - Epoch[14] Avg accuracy: 0.49 Avg loss: 0.77
End of Epoch 14: Learning Rate 0.0008
1.934782608695652 0.6742424242424242
Training Results - Epoch[1] Avg 

#### CALIB

In [30]:
#chosen nets: "large", 32/16

layers= [32,16]
hic_df_general_path= DATASETS[ds_name]['paths']['hic_df_save_path']
net_general_path= DATASETS[ds_name]['paths']['def_net_save_path']
# retrieving df and best net path
for i in range (1, 4):

    df_switch_path= os.path.join(hic_df_general_path,
                            f"iter_{i}",
                            f"results_{params['user_name']}",
                            f"hic_{params['user_name']}.parquet"
                            )
    
    df_switch= pd.read_parquet(df_switch_path)


    def_net_path= os.path.join(net_general_path,
                           f"iter_{i}",
                           f"{user_suffix}_models",
                           f"{layers[0]}_{layers[1]}"
                          )

    pattern = f"{layers[0]}_{layers[1]}_model_*.pt"
    search_query = os.path.join(def_net_path, pattern)
    matches = glob.glob(search_query)

    if matches:
        def_net_path = max(matches, key=os.path.getctime)
        run_calibration(def_net_path= def_net_path, 
                        df_switch=df_switch, 
                        ds_name=ds_name, 
                        params=params,
                        device=device, 
                        iteration=i,
                        def_net_layers=net_layers
                        ) #default had batch seize 128, epochs = 20 

#### MIC

In [31]:
current_user_name= "accurate_not_trusting"
user_suffix= 'acc_nt'

exp_path= fr".\trained_experts\{ds_name}\{name}.pkl"
current_expert= joblib.load(exp_path)


params = base_params.copy()  # Start with the base
params.update({
    "user_name": current_user_name,
    "user_suffix": user_suffix,
    "run_confidence": True,
    "run_mao": True
})

objects = base_objects.copy() # Start with the base clones
objects.update({
    "user_model": current_expert
})


def_net_layers= [32, 16]
strat_1=None
strat_2=None

for i in range(1,4):

    strat_1, strat_2= run_mic( 
                            ds_name= ds_name,
                            device= device,
                            layers= def_net_layers,
                            params=params,
                            objects=objects,
                            iteration=i,
                            strat_1_res=strat_1,
                            strat_2_res=strat_2)
    
  

100%|██████████| 1085/1085 [01:31<00:00, 11.85it/s]
c:\Bridget\orchestrator.py:953: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  anqi_mao_thresh= anqi_mao_thresh_row[0]
 25%|██▍       | 269/1085 [00:23<01:07, 12.05it/s]

2026-03-27 17:09:56,513 | WARNING | Drift here! Record index 270
2026-03-27 17:09:56,529 | WARNING | DRIFT DETECTED | Phase: MiC | Strat: Mao


 25%|██▍       | 270/1085 [00:23<01:11, 11.38it/s]
c:\Bridget\orchestrator.py:953: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  anqi_mao_thresh= anqi_mao_thresh_row[0]
 25%|██▍       | 269/1085 [00:23<01:11, 11.40it/s]

2026-03-27 17:10:20,336 | WARNING | Drift here! Record index 270
2026-03-27 17:10:20,356 | WARNING | DRIFT DETECTED | Phase: MiC | Strat: Mao


 25%|██▍       | 270/1085 [00:23<01:11, 11.35it/s]
c:\Bridget\orchestrator.py:953: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  anqi_mao_thresh= anqi_mao_thresh_row[0]
 25%|██▍       | 269/1085 [00:23<01:09, 11.80it/s]

...

 25%|██▍       | 270/1085 [00:26<01:19, 10.28it/s]
c:\Bridget\orchestrator.py:953: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  anqi_mao_thresh= anqi_mao_thresh_row[0]
 25%|██▍       | 269/1085 [00:26<01:09, 11.73it/s]

2026-03-27 17:13:43,214 | WARNING | Drift here! Record index 270
2026-03-27 17:13:43,228 | WARNING | DRIFT DETECTED | Phase: MiC | Strat: Mao


 25%|██▍       | 270/1085 [00:26<01:19, 10.26it/s]
c:\Bridget\orchestrator.py:953: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  anqi_mao_thresh= anqi_mao_thresh_row[0]
 25%|██▍       | 269/1085 [00:26<01:09, 11.82it/s]

2026-03-27 17:14:09,542 | WARNING | Drift here! Record index 270
2026-03-27 17:14:09,556 | WARNING | DRIFT DETECTED | Phase: MiC | Strat: Mao


100%|██████████| 1085/1085 [01:40<00:00, 10.80it/s]
c:\Bridget\orchestrator.py:953: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  anqi_mao_thresh= anqi_mao_thresh_row[0]
100%|██████████| 1085/1085 [01:39<00:00, 10.88it/s]
c:\Bridget\orchestrator.py:953: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  anqi_mao_thresh= anqi_mao_thresh_row[0]
100%|██████████| 1085/1085 [01:40<00:00, 10.82it/s]
c:\Bridget\orchestrator.py:953: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value b

### Expert: Inaccurate, Not Trusting 

#### HIC

In [32]:
#setting up fixed params

current_user_name= "inaccurate_not_trusting"
user_suffix= 'inacc_nt'

exp_path= fr".\trained_experts\{ds_name}\{name}.pkl"
current_expert= joblib.load(exp_path)


params = base_params.copy()  # Start with the base
params.update({
    "user_name": current_user_name,
    "user_suffix": user_suffix
})

objects = base_objects.copy() # Start with the base clones
objects.update({
    "user_model": current_expert
})

In [33]:
run_hic(ds_name, params, objects)

100%|██████████| 722/722 [01:30<00:00,  7.99it/s]
2026-03-27 17:25:36,134 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training


1.0749235474006116 0.9348404255319149
Training Results - Epoch[1] Avg accuracy: 0.53 Avg loss: 0.71
Validation Results - Epoch[1] Avg accuracy: 0.41 Avg loss: 0.74
End of Epoch 1: Learning Rate 0.001
Training Results - Epoch[2] Avg accuracy: 0.53 Avg loss: 0.70
Validation Results - Epoch[2] Avg accuracy: 0.41 Avg loss: 0.73
End of Epoch 2: Learning Rate 0.001
Training Results - Epoch[3] Avg accuracy: 0.53 Avg loss: 0.70
Validation Results - Epoch[3] Avg accuracy: 0.41 Avg loss: 0.73
End of Epoch 3: Learning Rate 0.001
Training Results - Epoch[4] Avg accuracy: 0.53 Avg loss: 0.70
Validation Results - Epoch[4] Avg accuracy: 0.41 Avg loss: 0.73
End of Epoch 4: Learning Rate 0.001


c:\Users\virgm\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\virgm\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\virgm\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


1.0749235474006116 0.9348404255319149
Training Results - Epoch[1] Avg accuracy: 0.53 Avg loss: 0.69
Validation Results - Epoch[1] Avg accuracy: 0.41 Avg loss: 0.71
End of Epoch 1: Learning Rate 0.001
Training Results - Epoch[2] Avg accuracy: 0.53 Avg loss: 0.69
Validation Results - Epoch[2] Avg accuracy: 0.41 Avg loss: 0.71
End of Epoch 2: Learning Rate 0.001
Training Results - Epoch[3] Avg accuracy: 0.54 Avg loss: 0.69
Validation Results - Epoch[3] Avg accuracy: 0.42 Avg loss: 0.70
End of Epoch 3: Learning Rate 0.001
Training Results - Epoch[4] Avg accuracy: 0.56 Avg loss: 0.69
Validation Results - Epoch[4] Avg accuracy: 0.47 Avg loss: 0.70
End of Epoch 4: Learning Rate 0.001
Training Results - Epoch[5] Avg accuracy: 0.59 Avg loss: 0.69
Validation Results - Epoch[5] Avg accuracy: 0.50 Avg loss: 0.69
End of Epoch 5: Learning Rate 0.001
Training Results - Epoch[6] Avg accuracy: 0.60 Avg loss: 0.68
Validation Results - Epoch[6] Avg accuracy: 0.53 Avg loss: 0.69
End of Epoch 6: Learning R

2026-03-27 17:25:37,312 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training


...

1.7536945812807883 0.6994106090373281
Training Results - Epoch[1] Avg accuracy: 0.71 Avg loss: 0.69
Validation Results - Epoch[1] Avg accuracy: 0.41 Avg loss: 0.74
End of Epoch 1: Learning Rate 0.001
Training Results - Epoch[2] Avg accuracy: 0.72 Avg loss: 0.68
Validation Results - Epoch[2] Avg accuracy: 0.41 Avg loss: 0.74
End of Epoch 2: Learning Rate 0.001
Training Results - Epoch[3] Avg accuracy: 0.74 Avg loss: 0.68
Validation Results - Epoch[3] Avg accuracy: 0.46 Avg loss: 0.73
End of Epoch 3: Learning Rate 0.001
Training Results - Epoch[4] Avg accuracy: 0.78 Avg loss: 0.67
Validation Results - Epoch[4] Avg accuracy: 0.49 Avg loss: 0.72
End of Epoch 4: Learning Rate 0.001
Training Results - Epoch[5] Avg accuracy: 0.80 Avg loss: 0.66
Validation Results - Epoch[5] Avg accuracy: 0.49 Avg loss: 0.72
End of Epoch 5: Learning Rate 0.001
Training Results - Epoch[6] Avg accuracy: 0.81 Avg loss: 0.66
Validation Results - Epoch[6] Avg accuracy: 0.49 Avg loss: 0.71
End of Epoch 6: Learning R

2026-03-27 17:28:58,281 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training


Validation Results - Epoch[9] Avg accuracy: 0.49 Avg loss: 0.71
End of Epoch 9: Learning Rate 0.001
1.7536945812807883 0.6994106090373281
Training Results - Epoch[1] Avg accuracy: 0.71 Avg loss: 0.69
Validation Results - Epoch[1] Avg accuracy: 0.41 Avg loss: 0.76
End of Epoch 1: Learning Rate 0.001
Training Results - Epoch[2] Avg accuracy: 0.71 Avg loss: 0.68
Validation Results - Epoch[2] Avg accuracy: 0.41 Avg loss: 0.76
End of Epoch 2: Learning Rate 0.001
Training Results - Epoch[3] Avg accuracy: 0.71 Avg loss: 0.67
Validation Results - Epoch[3] Avg accuracy: 0.41 Avg loss: 0.76
End of Epoch 3: Learning Rate 0.001
Training Results - Epoch[4] Avg accuracy: 0.76 Avg loss: 0.66
Validation Results - Epoch[4] Avg accuracy: 0.46 Avg loss: 0.76
End of Epoch 4: Learning Rate 0.001
Training Results - Epoch[5] Avg accuracy: 0.79 Avg loss: 0.65
Validation Results - Epoch[5] Avg accuracy: 0.48 Avg loss: 0.76
End of Epoch 5: Learning Rate 0.001
Training Results - Epoch[6] Avg accuracy: 0.80 Avg l

2026-03-27 17:28:58,888 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training


Training Results - Epoch[9] Avg accuracy: 0.81 Avg loss: 0.60
Validation Results - Epoch[9] Avg accuracy: 0.49 Avg loss: 0.78
End of Epoch 9: Learning Rate 0.0008
Training Results - Epoch[10] Avg accuracy: 0.81 Avg loss: 0.59
Validation Results - Epoch[10] Avg accuracy: 0.49 Avg loss: 0.78
End of Epoch 10: Learning Rate 0.0008
Training Results - Epoch[11] Avg accuracy: 0.81 Avg loss: 0.58
Validation Results - Epoch[11] Avg accuracy: 0.49 Avg loss: 0.79
End of Epoch 11: Learning Rate 0.0008
Training Results - Epoch[12] Avg accuracy: 0.81 Avg loss: 0.56
Validation Results - Epoch[12] Avg accuracy: 0.49 Avg loss: 0.79
End of Epoch 12: Learning Rate 0.0008
Training Results - Epoch[13] Avg accuracy: 0.81 Avg loss: 0.55
Validation Results - Epoch[13] Avg accuracy: 0.49 Avg loss: 0.79
End of Epoch 13: Learning Rate 0.0008
Training Results - Epoch[14] Avg accuracy: 0.81 Avg loss: 0.54
Validation Results - Epoch[14] Avg accuracy: 0.49 Avg loss: 0.79
End of Epoch 14: Learning Rate 0.0008
1.75369

#### CALIB

In [34]:
#chosen nets: "large", 32/16

layers= [32,16]
hic_df_general_path= DATASETS[ds_name]['paths']['hic_df_save_path']
net_general_path= DATASETS[ds_name]['paths']['def_net_save_path']

# retrieving df and best net path
for i in range (1, 4):

    df_switch_path= os.path.join(hic_df_general_path,
                            f"iter_{i}",
                            f"results_{params['user_name']}",
                            f"hic_{params['user_name']}.parquet"
                            )
    
    df_switch= pd.read_parquet(df_switch_path)


    def_net_path= os.path.join(net_general_path,
                           f"iter_{i}",
                           f"{user_suffix}_models",
                           f"{layers[0]}_{layers[1]}"
                          )

    pattern = f"{layers[0]}_{layers[1]}_model_*.pt"
    search_query = os.path.join(def_net_path, pattern)
    matches = glob.glob(search_query)

    if matches:
        def_net_path = max(matches, key=os.path.getctime)
        run_calibration(def_net_path= def_net_path, 
                        df_switch=df_switch, 
                        ds_name=ds_name, 
                        params=params,
                        device=device, 
                        iteration=i,
                        def_net_layers=net_layers
                        ) #default had batch seize 128, epochs = 20 

#### MIC

In [35]:
current_user_name= "inaccurate_not_trusting"
user_suffix= 'inacc_nt'

exp_path= fr".\trained_experts\{ds_name}\{name}.pkl"
current_expert= joblib.load(exp_path)


params = base_params.copy()  # Start with the base
params.update({
    "user_name": current_user_name,
    "user_suffix": user_suffix,
    "run_confidence": True,
    "run_mao": True
})

objects = base_objects.copy() # Start with the base clones
objects.update({
    "user_model": current_expert
})


def_net_layers= [32, 16]
strat_1=None
strat_2=None

for i in range(1,4):

    strat_1, strat_2= run_mic( 
                            ds_name= ds_name,
                            device= device,
                            layers= def_net_layers,
                            params=params,
                            objects=objects,
                            iteration=i,
                            strat_1_res=strat_1,
                            strat_2_res=strat_2)
    
  




 25%|██▍       | 269/1085 [00:22<01:10, 11.54it/s]

2026-03-27 17:29:28,565 | WARNING | Drift here! Record index 270
2026-03-27 17:29:28,583 | WARNING | DRIFT DETECTED | Phase: MiC | Strat: Confidence


 25%|██▍       | 270/1085 [00:23<01:09, 11.71it/s]
c:\Bridget\orchestrator.py:953: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  anqi_mao_thresh= anqi_mao_thresh_row[0]
 25%|██▍       | 269/1085 [00:23<01:11, 11.47it/s]

2026-03-27 17:29:51,898 | WARNING | Drift here! Record index 270
2026-03-27 17:29:51,914 | WARNING | DRIFT DETECTED | Phase: MiC | Strat: Mao


 25%|██▍       | 270/1085 [00:23<01:10, 11.59it/s]
c:\Bridget\orchestrator.py:953: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  anqi_mao_thresh= anqi_mao_thresh_row[0]
 25%|██▍       | 269/1085 [00:23<01:12, 11.25it/s]

...

 25%|██▍       | 270/1085 [00:25<01:16, 10.70it/s]
c:\Bridget\orchestrator.py:953: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  anqi_mao_thresh= anqi_mao_thresh_row[0]
 25%|██▍       | 269/1085 [00:24<01:07, 12.07it/s]

2026-03-27 17:35:11,144 | WARNING | Drift here! Record index 270
2026-03-27 17:35:11,163 | WARNING | DRIFT DETECTED | Phase: MiC | Strat: Mao


100%|██████████| 1085/1085 [01:42<00:00, 10.57it/s]
c:\Bridget\orchestrator.py:953: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  anqi_mao_thresh= anqi_mao_thresh_row[0]
100%|██████████| 1085/1085 [01:41<00:00, 10.66it/s]
c:\Bridget\orchestrator.py:953: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  anqi_mao_thresh= anqi_mao_thresh_row[0]
 25%|██▍       | 270/1085 [00:26<01:12, 11.27it/s]

2026-03-27 17:39:02,020 | WARNING | Drift here! Record index 270
2026-03-27 17:39:02,034 | WARNING | DRIFT DETECTED | Phase: MiC | Strat: Mao


 25%|██▍       | 270/1085 [00:26<01:19, 10.32it/s]
c:\Bridget\orchestrator.py:953: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  anqi_mao_thresh= anqi_mao_thresh_row[0]
100%|██████████| 1085/1085 [01:41<00:00, 10.64it/s]
c:\Bridget\orchestrator.py:953: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  anqi_mao_thresh= anqi_mao_thresh_row[0]
100%|██████████| 1085/1085 [01:42<00:00, 10.60it/s]
c:\Bridget\orchestrator.py:953: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by

## Benchmarking 

since COMPAS has their score already, lets test whether our experts are way off their prediction

### Human in Command

### Machine in Command (anche qui la 32/16 palesemente)

#### Experts re-calibration and prediction

In [36]:
# a note: in the BRIDGET pipeline, the Machine in Command phase receives the information provided by a scaler from the River lib

# however, since the ablation study will focus on a neural network that has to learn on the Human-in-Command portion anyways, 
# it needs its own dedicated scaler (a standard sklearn Min Max Scaler)

# now do the fitting again with the Min Max scaler, then get the predictions using the baseline func

set_all_seeds(42)

with open(fr".\experts_{ds_name}.yaml", "r") as f:
    config= yaml.safe_load(f)


params_dict= config['experts']['groups']['w_dict']

scaler= MinMaxScaler()
experts_obj= {}

X_scaled= scaler.fit_transform(X_hic_train)
X_scaled = pd.DataFrame(X_scaled, columns= X_hic_train.columns)

expert_names = ['accurate_trusting', 'accurate_not_trusting', 
                'inaccurate_trusting', 'inaccurate_not_trusting']

for name in expert_names:
    expert_type= config['experts']['groups'][name]

    experts_obj[name]= BetaUser(
        belief_level= expert_type['belief_value'],
        rethink_level= 0.8, # as suggested by the og FRANK implementation
        fairness= True,
        fpr= expert_type['target_FPR'],
        fnr= expert_type['target_FNR'],
        alpha= 0.9,
        features_dict= params_dict,
        seed= expert_type['group_seed']
        )
    res = experts_obj[name].fit(X_scaled, y_hic_train, tol= 0.001)

    save_dir= fr".\baseline\experts\{ds_name}"
    os.makedirs(save_dir, exist_ok= True)
    file_path = os.path.join(save_dir, f"{name}.pkl")

    with open(file_path, 'wb') as f:
        pickle.dump(experts_obj[name], f)
    print(f"Expert saved in: {file_path}")

    
    print(f"{'='*30}")
    print(f" EXPERT CALIBRATION REPORT ")
    print(f"{'='*30}")

    print(f"\n[EXPERT: {name}]")
    print(f"\n[FALSE POSITIVE RATE]")
    print(f"  - Iters:      {res['fpr iters number']}")
    print(f"  - Beta:       {res['calibrated_fpr_beta']:.4f}")
    print(f"  - Target:     {res['target_fpr']}")
    print(f"  - Achieved:   {res['achieved_fpr']:.4f}")

    print(f"\n[FALSE NEGATIVE RATE]")
    print(f"  - Iters:      {res['fnr iters number']}")
    print(f"  - Beta:       {res['calibrated_fnr_beta']:.4f}")
    print(f"  - Target:     {res['target_fnr']}")
    print(f"  - Achieved:   {res['achieved_fnr']:.4f}")
    


Expert saved in: .\baseline\experts\compas\accurate_trusting.pkl
 EXPERT CALIBRATION REPORT 

[EXPERT: accurate_trusting]

[FALSE POSITIVE RATE]
  - Iters:      12
  - Beta:       -3.1738
  - Target:     0.05
  - Achieved:   0.0509

[FALSE NEGATIVE RATE]
  - Iters:      11
  - Beta:       -2.6367
  - Target:     0.05
  - Achieved:   0.0504
Expert saved in: .\baseline\experts\compas\accurate_not_trusting.pkl
 EXPERT CALIBRATION REPORT 

[EXPERT: accurate_not_trusting]

[FALSE POSITIVE RATE]
  - Iters:      11
  - Beta:       -3.0273
  - Target:     0.05
  - Achieved:   0.0497

[FALSE NEGATIVE RATE]
  - Iters:      11
  - Beta:       -2.8320
  - Target:     0.05
  - Achieved:   0.0502
Expert saved in: .\baseline\experts\compas\inaccurate_trusting.pkl
 EXPERT CALIBRATION REPORT 

[EXPERT: inaccurate_trusting]

[FALSE POSITIVE RATE]
  - Iters:      14
  - Beta:       -1.2085
  - Target:     0.3
  - Achieved:   0.2999

[FALSE NEGATIVE RATE]
  - Iters:      14
  - Beta:       -0.4272
  - Tar

In [40]:
## since each strategy requires the label produced by the users, lets produce them :) 

## we'll use the calibrated experts in the (trained experts) folder 
# (alternatively, since you gotta rerun the preprocessing and calibration, just use the experts obj dict)
# to make it more proper, lets just retrieve the paths

experts_obj= {}
scaler= MinMaxScaler()
expert_names = ['accurate_trusting', 'accurate_not_trusting', 
                'inaccurate_trusting', 'inaccurate_not_trusting']

res= {}

for name in expert_names:

        res[name]= {}

        # retrieving trained exp
        exp_path= DATASETS[ds_name]['baseline_paths']['trained_experts']
        exp_path= os.path.join(exp_path, fr"{name}.pkl")
        trained_exp = joblib.load(exp_path)

        experts_obj[name]= trained_exp
        
        baseline_inst= Baseline(user=experts_obj[name],
                                label=DATASETS[ds_name]['target'],
                                train_set=dfs['hic_train'],
                                test_set=dfs['mic'],
                                val_set=dfs['calibration'],
                                x_train=X_hic_train,
                                y_train=y_hic_train,
                                x_test=X_mic,
                                y_test=y_mic,
                                x_val=X_val,
                                y_val=y_val
                                )
        
        train, test, val = baseline_inst.fit_expert(scaler)

        print(f"Train Predictions Unique Values: {train['expert prediction'].unique()}")
        
        print(f"Test Predictions Unique Values: {test['expert prediction'].unique()}")

        print(f"Val Predictions Unique Values: {val['expert prediction'].unique()}")

        res[name]['train']= train
        res[name]['test']= test
        res[name]['val']= val

        #################### SAVING STUFF

        ## 1. Train set
        train_path= DATASETS[ds_name]['baseline_paths']['train_df_labeled_by_experts']
        os.makedirs(train_path, exist_ok=True)
        file_path = os.path.join(train_path, f"train_baseline_{name}.parquet")

        train.to_parquet(file_path, index=False)
        
        ## 2. Test set
        test_path= DATASETS[ds_name]['baseline_paths']['test_df_for_mic']
        os.makedirs(test_path, exist_ok=True)
        file_path = os.path.join(test_path, f"test_baseline_{name}.parquet")

        test.to_parquet(file_path, index=False)

        ## 3. Validation set
        val_path= DATASETS[ds_name]['baseline_paths']['validation_set_labeled']
        os.makedirs(val_path, exist_ok=True)
        file_path = os.path.join(val_path, f"val_baseline_{name}.parquet")

        val.to_parquet(file_path, index=False)


        ## 4. Scaler 
        base_scaler_path= DATASETS[ds_name]['baseline_paths']['trained_scalers']
        os.makedirs(test_path, exist_ok=True)
        file_path = os.path.join(test_path, 
                                 f"mic",
                                 f"scaler_{name}.parquet")

        #################### TRAINING NN STRAIGHT AWAY
        log_dir = DATASETS[ds_name]['baseline_paths']['logs']
        user_log_path = os.path.join(log_dir, f"{name}.log")

        log= custom_log(name, user_log_path)
        for arch in ['small', 'medium', 'large', 'xl']:
                layers = NET_CONFIGS[arch]['layers']
                step_size= NET_CONFIGS[arch]['step_size']
                gamma= NET_CONFIGS[arch]['gamma']
                
                net_calibration(ds_name= ds_name,  
                        user_suffix= name, 
                        iter=None, 
                        target= DATASETS[ds_name]['target'] , 
                        train_set= train, 
                        val_set=val, 
                        feature_order=base_params['feature_order'], #without label 
                        net_layers=layers, 
                        net_params=DATASETS[ds_name]['net_params'], 
                        step_size=step_size,
                        gamma=gamma,
                        log=log,
                        device=device,
                        baseline= True)
        
        
        

        

100%|██████████| 1083/1083 [00:00<00:00, 13521.07it/s]


Train Predictions Unique Values: [1 0]
Test Predictions Unique Values: [1 0]
Val Predictions Unique Values: [0 1]
0.8445050662509743 1.2256787330316743
Training Results - Epoch[1] Avg accuracy: 0.41 Avg loss: 0.70
Validation Results - Epoch[1] Avg accuracy: 0.41 Avg loss: 0.70
End of Epoch 1: Learning Rate 0.001
Training Results - Epoch[2] Avg accuracy: 0.41 Avg loss: 0.70
Validation Results - Epoch[2] Avg accuracy: 0.41 Avg loss: 0.70
End of Epoch 2: Learning Rate 0.001


2026-03-27 23:30:19,070 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training


Training Results - Epoch[3] Avg accuracy: 0.41 Avg loss: 0.70
Validation Results - Epoch[3] Avg accuracy: 0.41 Avg loss: 0.70
End of Epoch 3: Learning Rate 0.001
Training Results - Epoch[4] Avg accuracy: 0.41 Avg loss: 0.69
Validation Results - Epoch[4] Avg accuracy: 0.41 Avg loss: 0.69
End of Epoch 4: Learning Rate 0.001


c:\Users\virgm\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\virgm\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\virgm\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


...

Training Results - Epoch[12] Avg accuracy: 0.63 Avg loss: 0.65
Validation Results - Epoch[12] Avg accuracy: 0.64 Avg loss: 0.64
End of Epoch 12: Learning Rate 0.0008
Training Results - Epoch[13] Avg accuracy: 0.63 Avg loss: 0.65
Validation Results - Epoch[13] Avg accuracy: 0.64 Avg loss: 0.64
End of Epoch 13: Learning Rate 0.0008
Training Results - Epoch[14] Avg accuracy: 0.63 Avg loss: 0.64
Validation Results - Epoch[14] Avg accuracy: 0.64 Avg loss: 0.64
End of Epoch 14: Learning Rate 0.0008
Training Results - Epoch[15] Avg accuracy: 0.63 Avg loss: 0.64
Validation Results - Epoch[15] Avg accuracy: 0.64 Avg loss: 0.64
End of Epoch 15: Learning Rate 0.0008
0.8445050662509743 1.2256787330316743
Training Results - Epoch[1] Avg accuracy: 0.41 Avg loss: 0.69
Validation Results - Epoch[1] Avg accuracy: 0.41 Avg loss: 0.69
End of Epoch 1: Learning Rate 0.001
Training Results - Epoch[2] Avg accuracy: 0.48 Avg loss: 0.69
Validation Results - Epoch[2] Avg accuracy: 0.48 Avg loss: 0.69
End of Epo

2026-03-27 23:30:33,239 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training


Training Results - Epoch[12] Avg accuracy: 0.63 Avg loss: 0.64
Validation Results - Epoch[12] Avg accuracy: 0.63 Avg loss: 0.64
End of Epoch 12: Learning Rate 0.0008
0.8445050662509743 1.2256787330316743
Training Results - Epoch[1] Avg accuracy: 0.61 Avg loss: 0.68
Validation Results - Epoch[1] Avg accuracy: 0.62 Avg loss: 0.68
End of Epoch 1: Learning Rate 0.001
Training Results - Epoch[2] Avg accuracy: 0.61 Avg loss: 0.67
Validation Results - Epoch[2] Avg accuracy: 0.61 Avg loss: 0.67
End of Epoch 2: Learning Rate 0.001
Training Results - Epoch[3] Avg accuracy: 0.62 Avg loss: 0.66
Validation Results - Epoch[3] Avg accuracy: 0.62 Avg loss: 0.66
End of Epoch 3: Learning Rate 0.001


2026-03-27 23:30:34,005 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training


Training Results - Epoch[4] Avg accuracy: 0.63 Avg loss: 0.65
Validation Results - Epoch[4] Avg accuracy: 0.64 Avg loss: 0.65
End of Epoch 4: Learning Rate 0.001
Training Results - Epoch[5] Avg accuracy: 0.63 Avg loss: 0.65
Validation Results - Epoch[5] Avg accuracy: 0.64 Avg loss: 0.64
End of Epoch 5: Learning Rate 0.001
Epoch[6], Iter[100] Loss: 0.66
Training Results - Epoch[6] Avg accuracy: 0.64 Avg loss: 0.65
Validation Results - Epoch[6] Avg accuracy: 0.63 Avg loss: 0.64
End of Epoch 6: Learning Rate 0.001
Training Results - Epoch[7] Avg accuracy: 0.64 Avg loss: 0.64
Validation Results - Epoch[7] Avg accuracy: 0.63 Avg loss: 0.64
End of Epoch 7: Learning Rate 0.001


In [41]:
base_params= {
    "cats": categoricals, #lst
    "num": numericals, #lss
    "warm_up_set": df_avv.copy(),
    "batch1": df_batch_1.copy(),
    "batch1_test": test_batch_1.copy(),
    "batch3":dfs['mic'].copy(),
    "validation_set": dfs['calibration'].copy(), #obtained by hic
    "feature_order": initial_ordering.copy(), #without the label ?
    "incremental_learner_name":"ARF",
    "n_cols": len(initial_ordering), #obtained by hic
    "mic_model_name": "Def_Net",
    "strat_1_name": "Confidence",
    "strat_2_name": "Mao",
    
}


learner_path= DATASETS[ds_name]['base_obj_paths']['incremental_learners']
path= os.path.join(learner_path, f"{base_params["incremental_learner_name"]}_model.pkl")
trained_learner= joblib.load(path)

prepr_path= DATASETS[ds_name]['base_obj_paths']['preprocessors']
path= os.path.join(prepr_path, f"{base_params["incremental_learner_name"]}_preprocessor.pkl")
base_prepr= joblib.load(path)

base_objects = {
      "preprocessor": copy.deepcopy(base_prepr), #or take from path
      "incremental_learner":copy.deepcopy(trained_learner), # or take from path
      "scaler": None
} 



In [ ]:
current_user_name= "accurate_trusting"
user_suffix= 'acc_t'

exp_path= fr".\trained_experts\{ds_name}\{name}.pkl"
current_expert= joblib.load(exp_path)


params = base_params.copy()  # Start with the base
params.update({
    "user_name": current_user_name,
    "user_suffix": user_suffix,
    "run_confidence": False,
    "run_mao": True
})

objects = base_objects.copy() # Start with the base clones
objects.update({
    "user_model": current_expert
})


def_net_layers= [32, 16]

batch3_path= DATASETS[ds_name]['paths']['scaled_mic_batch']
hic_df_general_path= DATASETS[ds_name]['paths']['hic_df_save_path']
mic_save_path = DATASETS[ds_name]['paths']['mic_df_save_path']



#### Tests

In [40]:
# retrieving assets 

current_dir = os.getcwd()
#repo_path = os.path.join(current_dir, "hdms-essai24", "L2Dcode")
repo_path = os.path.join(current_dir, "benchmark_helpers")
sys.path.append(repo_path)

from benchmark_helpers.lce_surrogate import LceSurrogate
from benchmark_helpers.basemethod import *

from benchmark_helpers.realizable_surrogate import RealizableSurrogate
from benchmark_helpers.metrics import *
from benchmark_helpers.functions import *

In [41]:
expert_names = ['accurate_trusting', 'accurate_not_trusting', 
                'inaccurate_trusting', 'inaccurate_not_trusting']

net_layers= [32,16]
arch= 'medium'
params= base_params.copy()
params.update({
    "human_preds_col_name": "expert prediction",
})


methods= ['CrossEntropy']

for method in methods: 
    run_benchmark(ds_name=ds_name, 
                  users_lst= expert_names, 
                  method=method,  
                  layers=net_layers, 
                  architecture=arch,
                  device= device,
                  params=params)

Computing L2D algorithm with method:  CrossEntropy
Retrieving Neural Network(s) models
Initializing the L2D algorithm
Fitting the model
loaded mode_dict (fit_hyperparam)


  0%|          | 0/3 [00:00<?, ?it/s]

Unique values in defers: [0 1]


  0%|          | 0/3 [00:01<?, ?it/s]


Unique values in defers: [1]


ValueError: Found empty input array (e.g., `y_true` or `y_pred`) while a minimum of 1 sample is required.

In [ ]:
experts_obj= {}
scaler= MinMaxScaler()
expert_names = ['accurate_trusting', 'accurate_not_trusting', 
                'inaccurate_trusting', 'inaccurate_not_trusting']

res= {}
layers=[32, 16]

for name in expert_names:

        res[name]= {}

        #################### RETRIEVING OBJECTS

        # 1. NET PATHS
        net_path= DATASETS[ds_name]['baseline_paths']['trained_nets']
        net_path= os.path.join(net_path,
                               f"{name}_models",
                               f"{layers[0]}_{layers[1]}")
        pattern = f"{layers[0]}_{layers[1]}_model_*.pt"
        search_query = os.path.join(net_path, pattern)
        matches = glob.glob(search_query)

        if matches:
                net_path = max(matches, key=os.path.getctime)

        # 2. TEST DATA
        test_data_path= DATASETS[ds_name]['baseline_paths']['test_df_for_mic']
        test_data_path= os.path.join(test_data_path, 
                                     f"test_baseline_{name}.parquet")
        test_d= pd.read_parquet(test_data_path)


        # 4. BENCHMARKING LCE SURROGATE
        method= 'Realizable'
        _, real_test, real_def_metrics, real_clf_metrics, real_coverage_metrics= get_l2d_results_and_metrics(method= method, 
                                input_size= base_params['n_cols'],
                                layers=layers,
                                n_classes=NET_CONFIGS['output_size'], 
                                test_data= test_d,
                                device= device, 
                                path=net_path
                                )
        
        res[name][method]['deferral_metrics']= real_def_metrics
        res[name][method]['clf_metrics']= real_clf_metrics
        res[name][method]['coverage_metrics']= real_coverage_metrics

        # SAVING RESULTS

        save_dir= DATASETS[ds_name]['baseline_paths']['benchmark_res']
        os.makedirs(save_dir, exist_ok=True)
        save_dir= os.path.join(save_dir,
                               f"{method}", f"res_{name}.parquet"        
                               )
        
        results_df = pd.DataFrame(res[name][method])
        results_df.to_parquet(save_dir, index=False)
        

        

